Установка зависимостей

In [ ]:

!pip install -q fastapi uvicorn transformers accelerate bitsandbytes torch sentencepiece peft
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.0 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏
added 22 packages in 4s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏npm notice
npm notice New major version of npm available! 10.8.2 -> 11.12.1
npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.12.1
npm notice To update run: npm install -g npm@11.12.1
npm notice
⠏

Импорты

In [ ]:

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn
from threading import Thread
import subprocess
import time
import asyncio
from concurrent.futures import ThreadPoolExecutor
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

Настройка пула потоков

In [ ]:
executor = ThreadPoolExecutor(max_workers=2)

Загрузка модели

In [ ]:

print("Загружаем модель по вашему методу...")
model = AutoModelForCausalLM.from_pretrained(
    "unsloth/Qwen2.5-7B-Instruct",
    load_in_4bit=True,
    device_map="auto",
    trust_remote_code=True,
    use_cache=False,  # ← Ваш параметр (отключает кэширование для экономии памяти)
)

tokenizer = AutoTokenizer.from_pretrained(
    "unsloth/Qwen2.5-7B-Instruct",
    trust_remote_code=True
)

# Загрузка вашего адаптера из локальной папки
model = PeftModel.from_pretrained(model, "./my-lora-adapter")
model.eval()  # Режим инференса

print(f"Модель загружена на: {model.device}")



Загружаем модель по вашему методу...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

Модель загружена на: cuda:0


FASTAPI

In [ ]:
from pydantic import BaseModel
from typing import List

app = FastAPI(title="Async Teacher LLM API")

class Message(BaseModel):
    role: str
    content: str

class GenerateRequest(BaseModel):
    messages: List[Message]
    max_new_tokens: int = 200

def _generate_sync(messages: list, max_new_tokens: int):
    try:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=3584,
            padding=False
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.1
            )

        response_tokens = outputs[0][inputs.input_ids.shape[1]:]
        response = tokenizer.decode(response_tokens, skip_special_tokens=True).strip()
        return response

    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        raise RuntimeError("Недостаточно памяти GPU. Уменьшите длину истории или max_new_tokens.")
    except Exception as e:
        error_msg = f"Ошибка генерации: {type(e).__name__}: {str(e)}"
        print(f"{error_msg}")
        raise RuntimeError(error_msg)

@app.post("/generate")
async def generate(request: GenerateRequest):
    """Принимает полную историю диалога и генерирует ответ"""
    if not request.messages:
        raise HTTPException(status_code=400, detail="История не может быть пустой")

    # Проверяем формат сообщений
    for msg in request.messages:
        if msg.role not in ["user", "assistant"]:
            raise HTTPException(status_code=400, detail="Роль должна быть 'user' или 'assistant'")
        if not msg.content.strip():
            raise HTTPException(status_code=400, detail="Сообщение не может быть пустым")

    if request.max_new_tokens < 1 or request.max_new_tokens > 500:
        raise HTTPException(status_code=400, detail="max_new_tokens должен быть от 1 до 500")

    try:
        loop = asyncio.get_event_loop()
        response = await loop.run_in_executor(
            executor,
            _generate_sync,
            [msg.dict() for msg in request.messages],  # Конвертируем в dict
            request.max_new_tokens
        )
        return {"response": response}

    except RuntimeError as e:
        raise HTTPException(status_code=500, detail=str(e))
    except Exception as e:
        raise HTTPException(status_code=503, detail=f"Сервис недоступен: {str(e)}")
@app.get("/health")
async def health_check():
    gpu_mem = torch.cuda.memory_allocated()/1024**3 if torch.cuda.is_available() else 0
    return {
        "status": "healthy",
        "model_loaded": True,
        "gpu_memory_gb": round(gpu_mem, 2),
        "device": str(model.device)
    }

Запуск

In [ ]:
# Запуск FastAPI в фоновом потоке
from threading import Thread
import time

def run_fastapi():
    uvicorn.run(app, host="127.0.0.1", port=8001, log_level="info")

# Запускаем сервер
server_thread = Thread(target=run_fastapi, daemon=True)
server_thread.start()

print("🚀 FastAPI сервер запущен в фоне")
print("Ожидание 3 секунды для инициализации...")
time.sleep(3)

# Проверка, что сервер работает
import requests
try:
    response = requests.get("http://localhost:8001/health")
    print("✅ Сервер ответил:", response.json())
except Exception as e:
    print("Сервер не отвечает:", str(e))

# Запуск localtunnel
import subprocess
lt_process = subprocess.Popen(
    ["lt", "--port", "8001"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Получение URL
import time
start_time = time.time()
while time.time() - start_time < 30:
    output = lt_process.stdout.readline()
    if output and "your url is:" in output:
        public_url = output.split("your url is:")[-1].strip()
        print(f"\nВаш URL: {public_url}")
        break
    time.sleep(0.5)

# ВАЖНО: Удерживаем ячейку активной!
print("\n🔄 Сервер работает. Не закрывайте эту ячейку!")
print("Для остановки нажмите кнопку 'Interrupt'")

# Бесконечный цикл для удержания ячейки
try:
    while True:
        time.sleep(60)
        # Периодическая проверка здоровья
        try:
            requests.get("http://localhost:8001/health")
        except:
            print("Сервер перестал отвечать")
except KeyboardInterrupt:
    print("\nОстановка сервера...")
    lt_process.terminate()